# 残基ごとの RSCC 計算（X-ray / Cryo-EM 対応）

PDB ID を入力として、RCSB から PDB を取得し、実験手法に応じて RSCC を計算します。

| 手法 | 密度データ | Phenix コマンド |
|------|-----------|----------------|
| X-ray | MTZ（Structure Factor から変換、またはローカル MTZ） | `phenix.real_space_correlation` |
| Cryo-EM | EMDB map（CCP4 `.map`） | `phenix.map_model_cc` |

**出力 CSV（ATOM / HETATM 分離）:**
- `{PDB_ID}_rscc_atom.csv`
- `{PDB_ID}_rscc_hetatm.csv`

列: `chain_id`, `chain_description`, `chain_residue_count`, `residue_number`, `residue_name`, `rscc`

> Cryo-EM では Phenix 1.20.1 の `real_space_correlation` が map 単体入力でエラーになるため、`map_model_cc` を使用します（残基ごとの CC を同様に取得）。

## Step 0: Python 依存パッケージ

初回のみ実行してください。

In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED = ["pandas", "matplotlib"]
missing = [pkg for pkg in REQUIRED if importlib.util.find_spec(pkg) is None]

if missing:
    print(f"Installing: {', '.join(missing)}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All required Python packages are already installed.")


## Step 1: 入力パラメータ

- `PDB_ID`: 4文字の PDB ID（例: `6xez`, `3klr`）
- `DATA_MODE`: `auto`（RCSB から自動判定）, `xray`, `cryoem`
- `LOCAL_MTZ_PATH` / `LOCAL_MAP_PATH`: ローカルファイルを直接指定する場合（省略可）
- `RESOLUTION`: Cryo-EM 用解像度（Å）。`None` なら RCSB から取得

In [ ]:
from pathlib import Path
from typing import Optional, Union

# ===== ここを変更 =====
PDB_ID = "6xez"
DATA_MODE = "auto"  # auto | xray | cryoem
LOCAL_MTZ_PATH: Optional[Union[str, Path]] = None
LOCAL_MAP_PATH: Optional[Union[str, Path]] = None
RESOLUTION: Optional[float] = None
# =====================

PDB_ID = PDB_ID.strip().lower()
if len(PDB_ID) != 4 or not PDB_ID.isalnum():
    raise ValueError(f"Invalid PDB ID: {PDB_ID!r}")

DATA_MODE = DATA_MODE.strip().lower()
if DATA_MODE not in {"auto", "xray", "cryoem"}:
    raise ValueError(f"Invalid DATA_MODE: {DATA_MODE!r}")

WORK_DIR = Path("work") / PDB_ID
WORK_DIR.mkdir(parents=True, exist_ok=True)

PDB_PATH = WORK_DIR / f"{PDB_ID}.pdb"
MTZ_PATH = Path(LOCAL_MTZ_PATH) if LOCAL_MTZ_PATH else WORK_DIR / f"{PDB_ID}.mtz"
SF_CIF_PATH = WORK_DIR / f"{PDB_ID}-sf.cif"
MAP_PATH = Path(LOCAL_MAP_PATH) if LOCAL_MAP_PATH else None
CSV_PATH_ATOM = WORK_DIR / f"{PDB_ID}_rscc_atom.csv"
CSV_PATH_HETATM = WORK_DIR / f"{PDB_ID}_rscc_hetatm.csv"

print(f"PDB ID     : {PDB_ID.upper()}")
print(f"DATA_MODE  : {DATA_MODE}")
print(f"Work dir   : {WORK_DIR.resolve()}")


## Step 2: 実行環境の確認

In [ ]:
import glob
import shutil
from typing import Optional


def find_executable(name: str, patterns: list[str]) -> Optional[str]:
    found = shutil.which(name)
    if found:
        return found
    for pattern in patterns:
        matches = sorted(glob.glob(pattern))
        if matches:
            return matches[-1]
    return None


PHENIX_RSCC = find_executable(
    "phenix.real_space_correlation",
    ["/Applications/phenix-*/build/bin/phenix.real_space_correlation"],
)
PHENIX_MAP_CC = find_executable(
    "phenix.map_model_cc",
    ["/Applications/phenix-*/build/bin/phenix.map_model_cc"],
)
GEMMI = find_executable(
    "gemmi",
    ["/Applications/ccp4-*/bin/gemmi", "/usr/local/bin/gemmi"],
)

print(f"phenix.real_space_correlation : {PHENIX_RSCC or 'NOT FOUND'}")
print(f"phenix.map_model_cc           : {PHENIX_MAP_CC or 'NOT FOUND'}")
print(f"gemmi                         : {GEMMI or 'NOT FOUND'}")

missing = []
if not PHENIX_RSCC:
    missing.append("phenix.real_space_correlation (X-ray)")
if not PHENIX_MAP_CC:
    missing.append("phenix.map_model_cc (Cryo-EM)")
if not GEMMI:
    missing.append("gemmi cif2mtz (X-ray)")
if missing:
    raise RuntimeError("Required tools not found:\n  - " + "\n  - ".join(missing))
print("\nAll required tools are available.")


## Step 3: RCSB から PDB をダウンロード

In [ ]:
import json
from urllib.error import HTTPError, URLError
from urllib.request import urlopen

RCSB_BASE_URL = "https://files.rcsb.org/download"
RCSB_ENTRY_URL = "https://data.rcsb.org/rest/v1/core/entry"


def download_file(url: str, output_path: Path, overwrite: bool = False) -> Path:
    if output_path.exists() and not overwrite:
        print(f"Skip (already exists): {output_path}")
        return output_path
    with urlopen(url, timeout=120) as response:
        data = response.read()
    output_path.write_bytes(data)
    print(f"Downloaded: {output_path} ({len(data):,} bytes)")
    return output_path


def fetch_entry_metadata(pdb_id: str) -> dict:
    url = f"{RCSB_ENTRY_URL}/{pdb_id.upper()}"
    with urlopen(url, timeout=60) as response:
        return json.loads(response.read().decode())


download_file(f"{RCSB_BASE_URL}/{PDB_ID}.pdb", PDB_PATH)
ENTRY_METADATA = fetch_entry_metadata(PDB_ID)

methods = [x.get("method", "") for x in ENTRY_METADATA.get("exptl", [])]
print(f"Experiment methods: {methods}")
print("\n--- PDB preview (first 5 lines) ---")
for line in PDB_PATH.read_text(errors="replace").splitlines()[:5]:
    print(line)


## Step 3.5: Chain metadata from RCSB

Fetch chain description and residue count from PDB entry metadata (RCSB REST API).
Falls back to residue counts parsed from the PDB file when metadata is unavailable.

In [ ]:
import pandas as pd

def fetch_rcsb_json(url: str) -> dict:
    with urlopen(url, timeout=60) as response:
        return json.loads(response.read().decode())


def count_pdb_residues_per_chain(
    pdb_path: Path, record_types: set[str] = {"ATOM"}
) -> dict[str, int]:
    residues: dict[str, set[tuple[str, str]]] = {}
    for line in pdb_path.read_text(errors="replace").splitlines():
        record = line[:6].strip()
        if record not in record_types:
            continue
        chain_id = line[21:22].strip() or "_"
        residue_key = (line[22:26].strip(), line[17:20].strip())
        residues.setdefault(chain_id, set()).add(residue_key)
    return {chain_id: len(keys) for chain_id, keys in residues.items()}


def fetch_chain_metadata(pdb_id: str, pdb_path: Path) -> pd.DataFrame:
    pdb_id_upper = pdb_id.upper()
    base = "https://data.rcsb.org/rest/v1/core"
    entry = fetch_rcsb_json(f"{base}/entry/{pdb_id_upper}")
    container = entry.get("rcsb_entry_container_identifiers", {})

    records: dict[str, dict] = {}

    for entity_id in container.get("polymer_entity_ids", []):
        entity = fetch_rcsb_json(f"{base}/polymer_entity/{pdb_id_upper}/{entity_id}")
        description = entity.get("rcsb_polymer_entity", {}).get("pdbx_description", "")
        residue_count = entity.get("entity_poly", {}).get("rcsb_sample_sequence_length")
        strand_ids = entity.get("entity_poly", {}).get("pdbx_strand_id", "")
        for chain_id in [s.strip() for s in strand_ids.replace(" ", "").split(",") if s.strip()]:
            records[chain_id] = {
                "chain_id": chain_id,
                "chain_description": description,
                "chain_residue_count": residue_count,
            }

    for entity_id in container.get("non_polymer_entity_ids", []):
        entity = fetch_rcsb_json(f"{base}/nonpolymer_entity/{pdb_id_upper}/{entity_id}")
        nonpoly = entity.get("pdbx_entity_nonpoly", {})
        description = (
            nonpoly.get("name")
            or entity.get("rcsb_nonpolymer_entity", {}).get("pdbx_description")
            or nonpoly.get("comp_id")
            or "Non-polymer entity"
        )
        chain_ids = entity.get("rcsb_nonpolymer_entity_container_identifiers", {}).get("asym_ids", [])
        for chain_id in chain_ids:
            records[chain_id] = {
                "chain_id": chain_id,
                "chain_description": description,
                "chain_residue_count": 1,
            }

    pdb_atom_counts = count_pdb_residues_per_chain(pdb_path, {"ATOM"})
    pdb_all_counts = count_pdb_residues_per_chain(pdb_path, {"ATOM", "HETATM"})
    all_chain_ids = sorted(set(records) | set(pdb_atom_counts) | set(pdb_all_counts))

    rows = []
    for chain_id in all_chain_ids:
        meta = records.get(chain_id, {})
        rows.append(
            {
                "chain_id": chain_id,
                "chain_description": meta.get("chain_description") or "Unknown",
                "chain_residue_count": meta.get("chain_residue_count")
                or pdb_atom_counts.get(chain_id)
                or pdb_all_counts.get(chain_id),
            }
        )
    return pd.DataFrame(rows)


CHAIN_METADATA = fetch_chain_metadata(PDB_ID, PDB_PATH)
print(CHAIN_METADATA.to_string(index=False))


## Step 4: 実験手法の判定と密度データの準備

X-ray → MTZ、Cryo-EM → EMDB map を取得します。

In [ ]:
import gzip
import shutil as sh
import subprocess


def resolve_data_mode(metadata: dict, mode: str) -> str:
    if mode in {"xray", "cryoem"}:
        return mode
    methods = " ".join(x.get("method", "").upper() for x in metadata.get("exptl", []))
    if "X-RAY" in methods:
        return "xray"
    if "ELECTRON MICROSCOPY" in methods or "ELECTRON CRYSTALLOGRAPHY" in methods:
        return "cryoem"
    raise ValueError(f"Cannot detect data mode from methods: {methods!r}. Set DATA_MODE manually.")


def get_resolution_angstrom(metadata: dict) -> Optional[float]:
    vals = metadata.get("rcsb_entry_info", {}).get("resolution_combined") or []
    return float(vals[0]) if vals else None


def get_primary_emdb_id(metadata: dict) -> str:
    ids = metadata.get("rcsb_entry_container_identifiers", {}).get("emdb_ids") or []
    if not ids:
        raise ValueError("No EMDB ID found in RCSB metadata.")
    return ids[0]


def download_emdb_map(emdb_id: str, output_map: Path) -> Path:
    emdb_num = emdb_id.upper().replace("EMD-", "")
    gz_path = output_map.with_suffix(output_map.suffix + ".gz")
    if output_map.exists():
        print(f"Skip (already exists): {output_map}")
        return output_map

    urls = [
        f"https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-{emdb_num}/map/emd_{emdb_num.lower()}.map.gz",
        f"https://data.pdbj.org/pub/emdb/structures/EMD-{emdb_num}/map/emd_{emdb_num.lower()}.map.gz",
    ]
    last_error = None
    for url in urls:
        try:
            download_file(url, gz_path, overwrite=True)
            break
        except Exception as exc:
            last_error = exc
            if gz_path.exists():
                gz_path.unlink()
    else:
        raise RuntimeError(f"Failed to download EMDB map for {emdb_id}: {last_error}")

    with gzip.open(gz_path, "rb") as src, open(output_map, "wb") as dst:
        sh.copyfileobj(src, dst)
    gz_path.unlink(missing_ok=True)
    print(f"Decompressed: {output_map} ({output_map.stat().st_size:,} bytes)")
    return output_map


RESOLVED_MODE = resolve_data_mode(ENTRY_METADATA, DATA_MODE)
print(f"Resolved data mode: {RESOLVED_MODE}")

if RESOLVED_MODE == "xray":
    if LOCAL_MTZ_PATH and Path(LOCAL_MTZ_PATH).exists():
        MTZ_PATH = Path(LOCAL_MTZ_PATH)
        print(f"Using local MTZ: {MTZ_PATH}")
    else:
        download_file(f"{RCSB_BASE_URL}/{PDB_ID}-sf.cif", SF_CIF_PATH)
        if not MTZ_PATH.exists():
            cmd = [GEMMI, "cif2mtz", str(SF_CIF_PATH), str(MTZ_PATH)]
            print("Command:", " ".join(cmd))
            result = subprocess.run(cmd, capture_output=True, text=True)
            if result.returncode != 0:
                raise RuntimeError(result.stderr or result.stdout)
            print(f"Created: {MTZ_PATH}")
        else:
            print(f"Skip (already exists): {MTZ_PATH}")
    DENSITY_PATH = MTZ_PATH
    PHENIX_TOOL = "real_space_correlation"
else:
    global MAP_PATH
    if MAP_PATH is None:
        emdb_id = get_primary_emdb_id(ENTRY_METADATA)
        MAP_PATH = WORK_DIR / f"{emdb_id.lower().replace('-', '_')}.map"
        print(f"Primary EMDB ID: {emdb_id}")
        download_emdb_map(emdb_id, MAP_PATH)
    elif not MAP_PATH.exists():
        raise FileNotFoundError(f"Local map not found: {MAP_PATH}")
    else:
        print(f"Using local map: {MAP_PATH}")

    if RESOLUTION is None:
        RESOLUTION = get_resolution_angstrom(ENTRY_METADATA)
    if RESOLUTION is None:
        raise ValueError("RESOLUTION (Å) is required for Cryo-EM. Set RESOLUTION in Step 1.")
    DENSITY_PATH = MAP_PATH
    PHENIX_TOOL = "map_model_cc"
    print(f"Resolution: {RESOLUTION} Å")

print(f"Density data : {DENSITY_PATH}")
print(f"Phenix tool  : {PHENIX_TOOL}")


## Step 5: Phenix で残基ごとの RSCC を計算

- **X-ray:** `phenix.real_space_correlation PDB MTZ detail=residue`
- **Cryo-EM:** `phenix.map_model_cc PDB MAP resolution=...` → `cc_per_residue.log`

※ 大きな構造（6XEZ 等）は数分かかることがあります。

In [ ]:
CC_PER_RESIDUE_LOG = WORK_DIR / "cc_per_residue.log"

if RESOLVED_MODE == "xray":
    cmd = [PHENIX_RSCC, str(PDB_PATH), str(MTZ_PATH), "detail=residue"]
    print("Command:", " ".join(cmd))
    print("Running...\n")
    result = subprocess.run(cmd, capture_output=True, text=True)
    PHENIX_STDOUT = result.stdout
    PHENIX_STDERR = result.stderr
    if result.returncode != 0:
        print(PHENIX_STDERR[-3000:] if len(PHENIX_STDERR) > 3000 else PHENIX_STDERR)
        raise RuntimeError(f"phenix.real_space_correlation failed (exit {result.returncode})")
    RSCC_OUTPUT = PHENIX_STDOUT
    RSCC_OUTPUT_FORMAT = "real_space_correlation"
else:
    cmd = [PHENIX_MAP_CC, str(PDB_PATH.resolve()), str(MAP_PATH.resolve()), f"resolution={RESOLUTION}"]
    print("Command:", " ".join(cmd))
    print("Running...\n")
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=WORK_DIR)
    PHENIX_STDOUT = result.stdout
    PHENIX_STDERR = result.stderr
    if result.returncode != 0:
        print(PHENIX_STDERR[-3000:] if len(PHENIX_STDERR) > 3000 else PHENIX_STDERR)
        print(PHENIX_STDOUT[-3000:] if len(PHENIX_STDOUT) > 3000 else PHENIX_STDOUT)
        raise RuntimeError(f"phenix.map_model_cc failed (exit {result.returncode})")
    if not CC_PER_RESIDUE_LOG.exists():
        raise FileNotFoundError(f"Expected log not found: {CC_PER_RESIDUE_LOG}")
    RSCC_OUTPUT = CC_PER_RESIDUE_LOG.read_text()
    RSCC_OUTPUT_FORMAT = "map_model_cc"

print(f"Output format: {RSCC_OUTPUT_FORMAT}")
print("\n--- Output tail (last 20 lines) ---")
for line in RSCC_OUTPUT.splitlines()[-20:]:
    print(line)


## Step 6: 出力をパースして DataFrame を作成

PDB の `ATOM` / `HETATM` 行で分類し、`df_atom` / `df_hetatm` を作成します。

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Tuple

import pandas as pd

TABLE_HEADER = "<id string>"
ResidueKey = Tuple[str, str, str]


@dataclass
class RsccRecord:
    chain_id: str
    residue_name: str
    residue_number: str
    rscc: float


def parse_pdb_residue_types(pdb_path: Path) -> Dict[ResidueKey, str]:
    residue_types: Dict[ResidueKey, str] = {}
    for line in pdb_path.read_text(errors="replace").splitlines():
        record = line[:6].strip()
        if record not in {"ATOM", "HETATM"}:
            continue
        key = (line[21:22].strip() or "_", line[22:26].strip(), line[17:20].strip())
        if key not in residue_types:
            residue_types[key] = record
        elif record == "ATOM" and residue_types[key] == "HETATM":
            residue_types[key] = "ATOM"
    return residue_types


def parse_real_space_correlation_output(stdout: str) -> List[RsccRecord]:
    records: List[RsccRecord] = []
    table_started = False
    for line in stdout.splitlines():
        if TABLE_HEADER in line:
            table_started = True
            continue
        if not table_started or not line.strip():
            continue
        parts = line.split()
        if len(parts) == 8:
            chain_id, residue_name, residue_number, _, _, rscc, _, _ = parts
        elif len(parts) == 9:
            chain_id, _, residue_name, residue_number, _, _, rscc, _, _ = parts
        else:
            continue
        try:
            records.append(RsccRecord(chain_id, residue_name, residue_number, float(rscc)))
        except ValueError:
            continue
    return records


def parse_map_model_cc_log(text: str) -> List[RsccRecord]:
    records: List[RsccRecord] = []
    for line in text.splitlines():
        parts = line.split()
        if len(parts) < 4:
            continue
        chain_id, residue_name, residue_number, rscc = parts[0], parts[1], parts[2], parts[3]
        if not residue_name.isalpha():
            continue
        try:
            records.append(RsccRecord(chain_id, residue_name, residue_number, float(rscc)))
        except ValueError:
            continue
    return records


if RSCC_OUTPUT_FORMAT == "real_space_correlation":
    records = parse_real_space_correlation_output(RSCC_OUTPUT)
else:
    records = parse_map_model_cc_log(RSCC_OUTPUT)

if not records:
    raise ValueError("No RSCC records parsed.")

df = pd.DataFrame({
    "chain_id": [r.chain_id for r in records],
    "residue_number": [r.residue_number for r in records],
    "residue_name": [r.residue_name for r in records],
    "rscc": [r.rscc for r in records],
})

df = df.merge(CHAIN_METADATA, on="chain_id", how="left")
df["chain_description"] = df["chain_description"].fillna("Unknown")

pdb_residue_types = parse_pdb_residue_types(PDB_PATH)
df["record_type"] = df.apply(
    lambda row: pdb_residue_types.get((row["chain_id"], str(row["residue_number"]), row["residue_name"]), "UNKNOWN"),
    axis=1,
)

df_atom = df[df["record_type"] == "ATOM"].copy()
df_hetatm = df[df["record_type"] == "HETATM"].copy()
df_unknown = df[df["record_type"] == "UNKNOWN"].copy()

print(f"Parsed {len(df)} records ({RSCC_OUTPUT_FORMAT})")
print(f"  ATOM   : {len(df_atom)}")
print(f"  HETATM : {len(df_hetatm)}")
if len(df_unknown):
    print(f"  UNKNOWN: {len(df_unknown)}")
print(f"\nChains: {sorted(df['chain_id'].unique())}")
print("\n--- ATOM: first 10 rows ---")
df_atom.head(10)


## Step 7: CSV 出力（ATOM / HETATM 分離）

In [ ]:
OUTPUT_COLUMNS = [
    "chain_id",
    "chain_description",
    "chain_residue_count",
    "residue_number",
    "residue_name",
    "rscc",
]


def save_rscc_csv(dataframe: pd.DataFrame, path: Path, label: str) -> None:
    dataframe[OUTPUT_COLUMNS].to_csv(path, index=False)
    print(f"Saved ({label}): {path.resolve()}  ({len(dataframe)} rows)")


save_rscc_csv(df_atom, CSV_PATH_ATOM, "ATOM")
save_rscc_csv(df_hetatm, CSV_PATH_HETATM, "HETATM")
if len(df_unknown):
    save_rscc_csv(df_unknown, WORK_DIR / f"{PDB_ID}_rscc_unknown.csv", "UNKNOWN")

print("\n--- ATOM CSV preview (first 5 lines) ---")
for line in CSV_PATH_ATOM.read_text().splitlines()[:6]:
    print(line)


## (Optional) Step 8: Matplotlib 日本語フォント設定

In [ ]:
import matplotlib
from matplotlib import font_manager


def configure_matplotlib_japanese() -> str:
    candidates = [
        "Hiragino Sans", "Hiragino Kaku Gothic ProN", "Hiragino Kaku Gothic Pro",
        "AppleGothic", "Arial Unicode MS", "Noto Sans CJK JP",
    ]
    available = {f.name for f in font_manager.fontManager.ttflist}
    for font_name in candidates:
        if font_name in available:
            matplotlib.rcParams["font.family"] = font_name
            matplotlib.rcParams["axes.unicode_minus"] = False
            return font_name
    matplotlib.rcParams["axes.unicode_minus"] = False
    return "default"


print(f"Matplotlib font: {configure_matplotlib_japanese()}")


## (Optional) Step 9: RSCC 分布（ATOM / HETATM 分離）

In [ ]:
import matplotlib.pyplot as plt

configure_matplotlib_japanese()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, data, title in [
    (axes[0], df_atom, f"ATOM ({PDB_ID.upper()})") ,
    (axes[1], df_hetatm, f"HETATM ({PDB_ID.upper()})") ,
]:
    if data.empty:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
    else:
        ax.hist(data["rscc"], bins=30, edgecolor="black", alpha=0.7)
        ax.axvline(data["rscc"].median(), color="red", linestyle="--", label=f"median={data['rscc'].median():.3f}")
        ax.legend()
    ax.set_xlabel("RSCC"); ax.set_ylabel("Count"); ax.set_title(title)
plt.tight_layout(); plt.show()


## (Optional) Step 10: 残基番号 vs RSCC（chain 別、ATOM / HETATM 分離）

In [ ]:
import matplotlib.pyplot as plt

configure_matplotlib_japanese()


def chain_legend_label(chain_id: str) -> str:
    row = CHAIN_METADATA[CHAIN_METADATA["chain_id"] == chain_id]
    if row.empty:
        return f"Chain {chain_id}"
    desc = row.iloc[0]["chain_description"]
    count = row.iloc[0]["chain_residue_count"]
    if pd.notna(count):
        return f"Chain {chain_id}: {desc} (n={int(count)})"
    return f"Chain {chain_id}: {desc}"


def plot_rscc_by_residue(dataframe: pd.DataFrame, title: str, ax) -> None:
    if dataframe.empty:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(title)
        return
    plot_df = dataframe.copy()
    plot_df["residue_num"] = pd.to_numeric(plot_df["residue_number"], errors="coerce")
    for chain_id, group in plot_df.groupby("chain_id", sort=True):
        group = group.sort_values(["residue_num", "residue_number"])
        ax.plot(
            group["residue_num"],
            group["rscc"],
            marker="o",
            markersize=2,
            linewidth=1,
            label=chain_legend_label(chain_id),
            alpha=0.85,
        )
    ax.set_xlabel("Residue number")
    ax.set_ylabel("RSCC")
    ax.set_title(title)
    ax.set_ylim(0, 1.05)
    ax.legend(title="Chain", fontsize=7, loc="best")
    ax.grid(True, alpha=0.3)


fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=False)
plot_rscc_by_residue(df_atom, f"ATOM - RSCC per residue ({PDB_ID.upper()})", axes[0])
plot_rscc_by_residue(df_hetatm, f"HETATM - RSCC per residue ({PDB_ID.upper()})", axes[1])
plt.tight_layout()
plt.show()


## (Optional) Step 11: Selected chains only — RSCC per residue

Specify chain ID(s) below and re-plot RSCC vs residue number for those chains only.

Requires Step 6+ (`df`, `df_atom`, `CHAIN_METADATA`). Step 10 does not need to be re-run.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager

# --- helpers (self-contained if Step 8/10 were skipped) ---
if "configure_matplotlib_japanese" not in globals():
    def configure_matplotlib_japanese() -> str:
        for font_name in [
            "Hiragino Sans", "Hiragino Kaku Gothic ProN", "AppleGothic",
            "Arial Unicode MS", "Noto Sans CJK JP",
        ]:
            if font_name in {f.name for f in font_manager.fontManager.ttflist}:
                matplotlib.rcParams["font.family"] = font_name
                matplotlib.rcParams["axes.unicode_minus"] = False
                return font_name
        matplotlib.rcParams["axes.unicode_minus"] = False
        return "default"

if "chain_legend_label" not in globals():
    def chain_legend_label(chain_id: str) -> str:
        row = CHAIN_METADATA[CHAIN_METADATA["chain_id"] == chain_id]
        if row.empty:
            return f"Chain {chain_id}"
        desc = row.iloc[0]["chain_description"]
        count = row.iloc[0]["chain_residue_count"]
        if pd.notna(count):
            return f"Chain {chain_id}: {desc} (n={int(count)})"
        return f"Chain {chain_id}: {desc}"

if "plot_rscc_by_residue" not in globals():
    def plot_rscc_by_residue(dataframe: pd.DataFrame, title: str, ax) -> None:
        if dataframe.empty:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
            ax.set_title(title)
            return
        plot_df = dataframe.copy()
        plot_df["residue_num"] = pd.to_numeric(plot_df["residue_number"], errors="coerce")
        for chain_id, group in plot_df.groupby("chain_id", sort=True):
            group = group.sort_values(["residue_num", "residue_number"])
            ax.plot(
                group["residue_num"], group["rscc"],
                marker="o", markersize=2, linewidth=1,
                label=chain_legend_label(chain_id), alpha=0.85,
            )
        ax.set_xlabel("Residue number")
        ax.set_ylabel("RSCC")
        ax.set_title(title)
        ax.set_ylim(0, 1.05)
        ax.legend(title="Chain", fontsize=7, loc="best")
        ax.grid(True, alpha=0.3)

configure_matplotlib_japanese()

required = {"df", "df_atom", "df_hetatm", "CHAIN_METADATA", "PDB_ID"}
missing = [name for name in required if name not in globals()]
if missing:
    raise NameError(
        f"Missing variable(s): {missing}. Run Step 1 and Step 3.5 through Step 6 first."
    )

# ===== ここを変更 =====
SELECTED_CHAINS = ["A", "E"]  # e.g. ["A"], ["A", "E", "F"]
PLOT_RECORD_TYPE = "atom"  # atom | hetatm | both
# =====================

available_chains = sorted(df["chain_id"].unique())
print("Available chains (in RSCC data):")
for chain_id in available_chains:
    row = CHAIN_METADATA[CHAIN_METADATA["chain_id"] == chain_id]
    if row.empty:
        print(f"  {chain_id}")
    else:
        r = row.iloc[0]
        print(f"  {chain_id}: {r['chain_description']} (n={r['chain_residue_count']})")

selected = [str(c).strip() for c in SELECTED_CHAINS if str(c).strip()]
invalid = [c for c in selected if c not in available_chains]
if invalid:
    raise ValueError(
        f"Unknown chain ID(s): {invalid}\n"
        f"Available in RSCC data: {available_chains}\n"
        f"Edit SELECTED_CHAINS in this cell."
    )
if not selected:
    raise ValueError("SELECTED_CHAINS is empty.")

plot_type = PLOT_RECORD_TYPE.strip().lower()
plot_specs = []
if plot_type in {"atom", "both"}:
    plot_specs.append((df_atom, "ATOM"))
if plot_type in {"hetatm", "both"}:
    plot_specs.append((df_hetatm, "HETATM"))
if not plot_specs:
    raise ValueError("PLOT_RECORD_TYPE must be 'atom', 'hetatm', or 'both'.")

print(f"\nSelected chains: {selected}")
chain_label = ", ".join(selected)

nrows = len(plot_specs)
fig, axes = plt.subplots(nrows, 1, figsize=(14, 4.5 * nrows))
if nrows == 1:
    axes = [axes]

for ax, (data, label) in zip(axes, plot_specs):
    filtered = data[data["chain_id"].isin(selected)].copy()
    title = f"{label} - RSCC per residue ({PDB_ID.upper()}) - chains {chain_label}"
    plot_rscc_by_residue(filtered, title, ax)

plt.tight_layout()
plt.show()

for data, label in plot_specs:
    n = len(data[data["chain_id"].isin(selected)])
    print(f"{label}: {n} rows plotted")


## (Optional) Step 12: Individual chain plots with highlighted residues

One RSCC-vs-residue plot per chain. Specified residues are shaded with a light red background.

Default highlights (6XEZ):
- Chain A: 901, 902, 903, 904
- Chain E, F: 92, 93, 94, 95

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager

if "configure_matplotlib_japanese" not in globals():
    def configure_matplotlib_japanese() -> str:
        for font_name in [
            "Hiragino Sans", "Hiragino Kaku Gothic ProN", "AppleGothic",
            "Arial Unicode MS", "Noto Sans CJK JP",
        ]:
            if font_name in {f.name for f in font_manager.fontManager.ttflist}:
                matplotlib.rcParams["font.family"] = font_name
                matplotlib.rcParams["axes.unicode_minus"] = False
                return font_name
        matplotlib.rcParams["axes.unicode_minus"] = False
        return "default"

if "chain_legend_label" not in globals():
    def chain_legend_label(chain_id: str) -> str:
        row = CHAIN_METADATA[CHAIN_METADATA["chain_id"] == chain_id]
        if row.empty:
            return f"Chain {chain_id}"
        desc = row.iloc[0]["chain_description"]
        count = row.iloc[0]["chain_residue_count"]
        if pd.notna(count):
            return f"Chain {chain_id}: {desc} (n={int(count)})"
        return f"Chain {chain_id}: {desc}"

configure_matplotlib_japanese()

required = {"df_atom", "CHAIN_METADATA", "PDB_ID"}
missing = [name for name in required if name not in globals()]
if missing:
    raise NameError(
        f"Missing variable(s): {missing}. Run Step 1 and Step 3.5 through Step 6 first."
    )

# ===== ここを変更 =====
HIGHLIGHT_RESIDUES = {
    "A": [901, 902, 903, 904],
    "E": [92, 93, 94, 95],
    "F": [92, 93, 94, 95],
}
PLOT_CHAINS = list(HIGHLIGHT_RESIDUES.keys())  # or e.g. ["A", "E"]
HIGHLIGHT_COLOR = "#ffcccc"
HIGHLIGHT_ALPHA = 0.55
# =====================


def add_residue_highlights(ax, residue_numbers, color=HIGHLIGHT_COLOR, alpha=HIGHLIGHT_ALPHA) -> None:
    """Shade each highlighted residue number on the x-axis."""
    for resnum in sorted({int(r) for r in residue_numbers}):
        ax.axvspan(resnum - 0.5, resnum + 0.5, color=color, alpha=alpha, zorder=0)


def plot_single_chain_rscc(
    chain_id: str,
    dataframe: pd.DataFrame,
    highlight_residues: list,
    ax,
) -> None:
    chain_df = dataframe[dataframe["chain_id"] == chain_id].copy()
    if chain_df.empty:
        ax.text(0.5, 0.5, f"No data for chain {chain_id}", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(chain_legend_label(chain_id))
        return

    add_residue_highlights(ax, highlight_residues)

    chain_df["residue_num"] = pd.to_numeric(chain_df["residue_number"], errors="coerce")
    chain_df = chain_df.sort_values(["residue_num", "residue_number"])
    ax.plot(
        chain_df["residue_num"],
        chain_df["rscc"],
        marker="o",
        markersize=2,
        linewidth=1,
        color="tab:blue",
        alpha=0.9,
        zorder=2,
        label="RSCC",
    )

    highlight_label = ", ".join(str(r) for r in sorted(highlight_residues))
    ax.plot([], [], color=HIGHLIGHT_COLOR, alpha=HIGHLIGHT_ALPHA, linewidth=8, label=f"Highlight: {highlight_label}")

    ax.set_xlabel("Residue number")
    ax.set_ylabel("RSCC")
    ax.set_title(f"{chain_legend_label(chain_id)} ({PDB_ID.upper()})")
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8, loc="best")
    ax.grid(True, alpha=0.3, zorder=1)


available_chains = sorted(df_atom["chain_id"].unique())
plot_chains = [str(c).strip() for c in PLOT_CHAINS if str(c).strip()]
invalid = [c for c in plot_chains if c not in available_chains]
if invalid:
    raise ValueError(f"Chain(s) not in ATOM data: {invalid}. Available: {available_chains}")

nrows = len(plot_chains)
fig, axes = plt.subplots(nrows, 1, figsize=(14, 4.2 * nrows))
if nrows == 1:
    axes = [axes]

for ax, chain_id in zip(axes, plot_chains):
    highlights = HIGHLIGHT_RESIDUES.get(chain_id, [])
    if not highlights:
        print(f"Warning: no highlights defined for chain {chain_id}")
    plot_single_chain_rscc(chain_id, df_atom, highlights, ax)

plt.tight_layout()
plt.show()
